# Model Comparison
Loads all new-format artifacts from `artifacts/` and compares CV performance across models, horizons, and training modes.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from config import ARTIFACTS_PATH

sns.set_theme(style='whitegrid', palette='tab10', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120

## 1  Load artifacts

In [ ]:
def load_artifacts(path: pathlib.Path) -> dict:
    """Load all new-format artifacts (those with cv_summary) keyed by filename stem."""
    arts = {}
    for f in sorted(path.glob('*.pkl')):
        a = joblib.load(f)
        if 'cv_summary' not in a:          # skip old-format artifacts
            print(f'  skip (old format): {f.name}')
            continue
        arts[f.stem] = a
        print(f'  loaded: {f.stem:20s}  h={a["horizon"]}  mode={a["mode"]}  '
              f'ft={a["ft_type"]}  windows={len(a["cv_metrics"])}')
    print(f'\n{len(arts)} artifact(s) loaded.')
    return arts

artifacts = load_artifacts(ARTIFACTS_PATH)

In [ ]:
# Build a flat summary DataFrame  (one row per artifact)
METRICS = ['balanced_accuracy', 'roc_auc', 'log_loss', 'mcc', 'accuracy']

rows = []
for stem, a in artifacts.items():
    row = {
        'name':    stem,
        'model':   a['model_key'],
        'horizon': a['horizon'],
        'mode':    a['mode'],
        'ft_type': a['ft_type'],
        'windows': len(a['cv_metrics']),
        'train_start': a['train_start'],
        'train_end':   a['train_end'],
    }
    for m in METRICS:
        s = a['cv_summary'].get(m, {})
        row[f'{m}_mean'] = s.get('mean', np.nan)
        row[f'{m}_std']  = s.get('std',  np.nan)
    rows.append(row)

summary = pd.DataFrame(rows).set_index('name')

# Also build a long DataFrame with one row per (artifact, cv_window)
window_rows = []
for stem, a in artifacts.items():
    for wi, m in enumerate(a['cv_metrics']):
        window_rows.append({
            'name':    stem,
            'model':   a['model_key'],
            'horizon': a['horizon'],
            'mode':    a['mode'],
            'window':  wi,
            **{k: v for k, v in m.items() if k in METRICS},
        })
windows_df = pd.DataFrame(window_rows)

print('Summary shape:', summary.shape)
print('Windows long df shape:', windows_df.shape)

## 2  CV Metrics Summary Table

In [ ]:
def fmt(mean, std):
    return f'{mean:.4f} ± {std:.4f}'

display_cols = ['model', 'horizon', 'mode', 'ft_type', 'windows']
for m in METRICS:
    summary[m] = summary.apply(lambda r: fmt(r[f'{m}_mean'], r[f'{m}_std']), axis=1)

display_df = summary[display_cols + METRICS].sort_values(
    ['horizon', 'balanced_accuracy_mean'], ascending=[True, False]
)

display_df[display_cols + METRICS].style \
    .set_caption('CV metrics  (mean ± std across sliding/expanding windows)') \
    .set_table_styles([{'selector': 'caption', 'props': 'font-size:13px; font-weight:bold;'}])

## 3  Per-Window Distributions  (boxplots)

In [ ]:
def boxplot_metric(df: pd.DataFrame, metric: str, ax, title_suffix=''):
    order = (
        df.groupby('name')[metric].median()
          .sort_values(ascending=False).index.tolist()
    )
    palette = {n: sns.color_palette('tab10')[i % 10] for i, n in enumerate(order)}
    sns.boxplot(data=df, x=metric, y='name', order=order,
                palette=palette, width=0.55, linewidth=0.9,
                flierprops=dict(marker='x', markersize=4), ax=ax)
    ax.set_title(f'{metric.replace("_", " ").title()}{title_suffix}', fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('')
    if metric == 'balanced_accuracy':
        ax.axvline(0.5, color='red', lw=0.8, ls='--', label='random (0.50)')
        ax.legend(fontsize=8)

primary_metrics = ['balanced_accuracy', 'roc_auc', 'mcc', 'log_loss']
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, m in zip(axes.flat, primary_metrics):
    boxplot_metric(windows_df, m, ax)

plt.suptitle('Per-window CV distribution by model', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4  Balanced Accuracy over CV Windows  (temporal trend)

In [ ]:
for horizon in sorted(windows_df['horizon'].unique()):
    df_h = windows_df[windows_df['horizon'] == horizon]
    fig, ax = plt.subplots(figsize=(13, 4))

    for name, grp in df_h.groupby('name'):
        ax.plot(grp['window'], grp['balanced_accuracy'], marker='o',
                markersize=4, lw=1.5, label=name)

    ax.axhline(0.5, color='red', lw=0.8, ls='--', label='random baseline')
    ax.set_title(f'Balanced accuracy per CV window — h={horizon}', fontsize=12)
    ax.set_xlabel('CV window index')
    ax.set_ylabel('Balanced accuracy')
    ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.3f'))
    ax.legend(loc='upper right', fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()

## 5  Head-to-Head Win Rate
For each pair of models (same horizon + mode), count in how many CV windows one beats the other on balanced accuracy.

In [ ]:
for (horizon, mode), grp in windows_df.groupby(['horizon', 'mode']):
    names = grp['name'].unique()
    if len(names) < 2:
        continue

    pivot = grp.pivot(index='window', columns='name', values='balanced_accuracy')
    n = len(names)
    win_matrix = pd.DataFrame(np.nan, index=names, columns=names)

    for a in names:
        for b in names:
            if a == b:
                continue
            valid = pivot[[a, b]].dropna()
            if len(valid):
                win_matrix.loc[a, b] = (valid[a] > valid[b]).mean()

    fig, ax = plt.subplots(figsize=(max(6, n * 1.2), max(4, n)))
    sns.heatmap(win_matrix.astype(float), annot=True, fmt='.2f', cmap='RdYlGn',
                vmin=0, vmax=1, linewidths=0.5, ax=ax,
                cbar_kws={'label': 'win rate (row beats col)'})
    ax.set_title(f'Win rate — h={horizon}, mode={mode}', fontsize=12)
    plt.tight_layout()
    plt.show()

## 6  Feature Importances  (tree models only)

In [ ]:
tree_arts = {k: v for k, v in artifacts.items() if 'feature_importances' in v}

if not tree_arts:
    print('No tree artifacts with feature importances found.')
else:
    ncols = min(2, len(tree_arts))
    nrows = (len(tree_arts) + 1) // 2
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 5 * nrows),
                             squeeze=False)

    for ax, (name, a) in zip(axes.flat, tree_arts.items()):
        imps = pd.Series(a['feature_importances']).sort_values(ascending=True).tail(20)
        imps.plot.barh(ax=ax, color='steelblue', edgecolor='none')
        ax.set_title(f'{name} — top 20 features (gain)', fontsize=10)
        ax.set_xlabel('Importance')
        ax.tick_params(axis='y', labelsize=8)

    # hide unused subplots
    for ax in axes.flat[len(tree_arts):]:
        ax.set_visible(False)

    plt.suptitle('Feature importances — tree models', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

## 7  Model Metadata

In [ ]:
meta_rows = []
for stem, a in artifacts.items():
    row = {
        'artifact':    stem,
        'model':       a['model_key'],
        'h':           a['horizon'],
        'mode':        a['mode'],
        'ft_type':     a['ft_type'],
        'train_start': a['train_start'],
        'train_end':   a['train_end'],
        'window_days': a.get('window_days', '?'),
        'n_features':  len(a.get('features', [])),
        'cv_windows':  len(a['cv_metrics']),
    }
    # tree-specific
    if 'n_rounds_final' in a:
        row['extra'] = f'n_rounds={a["n_rounds_final"]}'
    # neural-specific
    elif 'model_config' in a:
        cfg = a['model_config']
        parts = [f'cell={cfg.get("cell","?")}',
                 f'hidden={cfg.get("hidden_size","?")}',
                 f'seq={a.get("seq_len","?")}',
        ]
        if 'num_filters' in cfg:
            parts.append(f'filters={cfg["num_filters"]}')
        row['extra'] = '  '.join(parts)
    else:
        row['extra'] = ''
    meta_rows.append(row)

meta_df = pd.DataFrame(meta_rows).set_index('artifact')
meta_df.style.set_caption('Artifact metadata').set_table_styles(
    [{'selector': 'caption', 'props': 'font-size:13px; font-weight:bold;'}]
)

## 8  Metric Correlation across Windows

In [ ]:
if len(windows_df) > 0:
    corr = windows_df[METRICS].corr()
    fig, ax = plt.subplots(figsize=(6, 5))
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
                vmin=-1, vmax=1, mask=mask, linewidths=0.5, ax=ax)
    ax.set_title('Metric correlations (all models & windows)', fontsize=11)
    plt.tight_layout()
    plt.show()

## 9  Sliding vs Expanding — direct comparison
Only shown when the same model was trained under both modes.

In [ ]:
# Find models that appear in both sliding and expanding
s_df = summary[summary['mode'] == 'sliding']
e_df = summary[summary['mode'] == 'expanding']
common = set(s_df['model']) & set(e_df['model'])

if not common:
    print('No model has been trained under both modes yet.')
else:
    compare_rows = []
    for model in sorted(common):
        for m in ['balanced_accuracy', 'roc_auc', 'mcc']:
            s_val = s_df[s_df['model'] == model][f'{m}_mean'].values[0]
            e_val = e_df[e_df['model'] == model][f'{m}_mean'].values[0]
            compare_rows.append({'model': model, 'metric': m,
                                  'sliding': s_val, 'expanding': e_val,
                                  'delta (exp-sli)': e_val - s_val})

    comp_df = pd.DataFrame(compare_rows)
    comp_df.style \
        .background_gradient(subset=['delta (exp-sli)'], cmap='RdYlGn', vmin=-0.02, vmax=0.02) \
        .format({'sliding': '{:.4f}', 'expanding': '{:.4f}', 'delta (exp-sli)': '{:+.4f}'}) \
        .set_caption('Sliding vs Expanding — mean CV metrics')